<a href="https://colab.research.google.com/github/mona0101/Final-year-project-2026/blob/main/HDF5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#This notebook creates the three HDF5 files: train, test, and val. Do not run this code

In [ ]:
!git clone https://github.com/mona0101/Final-year-project-2026.git
%cd Final-year-project-2026

import os
from google.colab import drive
drive.mount('/content/drive')
# 1. Define Source and Destination
# Ensure these folder names match your Drive exactly!
DRIVE_FOLDER = '/content/drive/MyDrive/Colab Notebooks'
LOCAL_TARGET = '/content/drone_dataset'

if not os.path.exists(LOCAL_TARGET):
    os.makedirs(LOCAL_TARGET)

# 2. Use rsync to copy folders in parallel
# -a: archive mode (preserves everything)
# -v: verbose (shows progress)
# -h: human-readable
print("🚀 Starting high-speed parallel copy... this might take a few minutes but is faster than zipping.")

!rsync -ah --progress "{DRIVE_FOLDER}/Audio" "{LOCAL_TARGET}/"
!rsync -ah --progress "{DRIVE_FOLDER}/Video" "{LOCAL_TARGET}/"
!rsync -ah --progress "{DRIVE_FOLDER}/RF_Spectrograms" "{LOCAL_TARGET}/"

print(f"✅ All folders moved to {LOCAL_TARGET}")

Cloning into 'Final-year-project-2026'...
remote: Enumerating objects: 79, done.
remote: Counting objects: 100% (79/79), done.
remote: Compressing objects: 100% (77/77), done.
remote: Total 79 (delta 38), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (79/79), 718.68 KiB | 11.41 MiB/s, done.
Resolving deltas: 100% (38/38), done.
/content/Final-year-project-2026
Mounted at /content/drive
🚀 Starting high-speed parallel copy... this might take a few minutes but is faster than zipping.
sending incremental file list
Audio -> /content/drive/.shortcut-targets-by-id/1_DDNngF4WEG2Z6pQh8wtcd1M0ghq7K0i/Audio
sending incremental file list
Video -> /content/drive/.shortcut-targets-by-id/1GUJY-w67Bh_uN8FzbHGR26rw_2xOdKsE/Video
sending incremental file list
RF_Spectrograms -> /content/drive/.shortcut-targets-by-id/1ImiDL8BmTC_rNPVxEYSBaq9skoFEgZG5/RF_Spectrograms
✅ All folders moved to /content/drone_dataset


In [ ]:
import h5py
import numpy as np
import os
import librosa
import torch
import shutil
from tqdm import tqdm
from PIL import Image
from concurrent.futures import ProcessPoolExecutor
from google.colab import files  # لإتاحة تحميل الملفات للكمبيوتر

# --- 1. WORKER FUNCTION ---
def process_single_sample(args):
    sample, save_sr = args
    try:
        # AUDIO: Load 0.25s and force length to 11025
        y, _ = librosa.load(sample['audio_path'], sr=save_sr,
                            offset=sample['start'], duration=0.25)
        if len(y) < 11025:
            y = np.pad(y, (0, 11025 - len(y)))
        else:
            y = y[:11025]

        # VIDEO: Strict 7-frame check + LANCZOS Resize
        v_frames = []
        for p in sample['video_frame_paths']:
            if os.path.exists(p):
                img = Image.open(p).convert("RGB").resize((224, 224), Image.Resampling.LANCZOS)
                v_frames.append(np.array(img))
        if len(v_frames) != 7: return None
        v_data = np.stack(v_frames)

        # RF: Strict 1-frame check + LANCZOS Resize
        rf_frames = []
        for p in sample['rf_frame_paths']:
            if os.path.exists(p):
                img = Image.open(p).convert("RGB").resize((224, 224), Image.Resampling.LANCZOS)
                rf_frames.append(np.array(img))
        if len(rf_frames) != 1: return None
        rf_data = np.stack(rf_frames)

        return {
            'audio': y.astype(np.float16),
            'video': v_data.astype(np.uint8),
            'rf': rf_data.astype(np.uint8),
            'label': int(sample['label'])
        }
    except Exception:
        return None

# --- 2. PACKER FUNCTION (Modified for Local Save & PC Download) ---
def create_strict_h5(dataset_object, split_name):
    SAVE_SR = 44100
    NUM_WORKERS = os.cpu_count()
    # الحفظ محلياً في كولاب لتجنب امتلاء الدرايف
    local_path = f"/content/Drone_{split_name}.h5"

    tasks = [(dataset_object.samples[i], SAVE_SR) for i in range(len(dataset_object))]
    print(f"\n🚀 Packing {split_name} ({len(tasks)} samples) to Local Disk...")

    with ProcessPoolExecutor(max_workers=NUM_WORKERS) as executor:
        results = [r for r in list(tqdm(executor.map(process_single_sample, tasks, chunksize=10), total=len(tasks))) if r is not None]

    num_valid = len(results)
    if num_valid == 0:
        print(f"⚠️ No valid samples found for {split_name}. Skipping.")
        return

    with h5py.File(local_path, 'w') as f:
        f.create_dataset('audio', (num_valid, 11025), dtype='f2')
        f.create_dataset('video', (num_valid, 7, 224, 224, 3), dtype='u1')
        f.create_dataset('rf', (num_valid, 1, 224, 224, 3), dtype='u1')
        f.create_dataset('labels', (num_valid,), dtype='i1')
        f.attrs['sr'] = SAVE_SR

        for i, res in enumerate(results):
            f['audio'][i] = res['audio']
            f['video'][i] = res['video']
            f['rf'][i] = res['rf']
            f['labels'][i] = res['label']

    print(f"✅ Created Locally: {local_path} ({os.path.getsize(local_path)/(1024**2):.2f} MB)")

    # تحميل الملف لجهاز الكمبيوتر فوراً
    print(f"📥 Starting download of Drone_{split_name}.h5 to your PC...")
    files.download(local_path)

# --- 3. EXECUTION ---
# تأكد أن البيانات موجودة في هذا المسار المحلي أولاً
dataset_dir = '/content/drone_dataset'
audio_root = os.path.join(dataset_dir, 'Audio')
video_root = os.path.join(dataset_dir, 'Video')
rf_root    = os.path.join(dataset_dir, 'RF_Spectrograms')

# استيراد الـ Loader الخاص بك
from dataset_loader2 import DroneFusionDataset

print("🔍 Scanning folders...")
train_ds = DroneFusionDataset(audio_root+'/Train', video_root+'/Train', rf_root+'/Train', 'Train', audio_sr=44100)
val_ds   = DroneFusionDataset(audio_root+'/Validation', video_root+'/Validation', rf_root+'/Validation', 'Validation', audio_sr=44100)
test_ds  = DroneFusionDataset(audio_root+'/Test', video_root+'/Test', rf_root+'/Test', 'Test', audio_sr=44100)

# تنفيذ المعالجة والتحميل (بالترتيب)
# ملاحظة: سيظهر لك نافذة تحميل في المتصفح لكل ملف عند انتهائه
create_strict_h5(train_ds, "Train")
create_strict_h5(val_ds,   "Validation")
create_strict_h5(test_ds,  "Test")

🔍 Scanning folders...

🚀 Packing Train (8480 samples) to Local Disk...


100%|██████████| 8480/8480 [48:36<00:00,  2.91it/s]


✅ Created Locally: /content/Drone_Train.h5 (9917.08 MB)
📥 Starting download of Drone_Train.h5 to your PC...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...


🚀 Packing Test (1320 samples) to Local Disk...


100%|██████████| 1320/1320 [07:30<00:00,  2.93it/s]


✅ Created Locally: /content/Drone_Test.h5 (1543.70 MB)
📥 Starting download of Drone_Test.h5 to your PC...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>